In [3]:
!pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 15.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longe

In [4]:

from google.colab import files
uploaded = files.upload()

Saving preprocessed_multiple_class.csv to preprocessed_multiple_class.csv


In [5]:
import pandas as pd
bc_data = pd.read_csv("preprocessed_multiple_class.csv")

In [6]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

target = "attack type"

train_df, test_df = train_test_split(
    bc_data,
    test_size=0.25,
    random_state=42,
    stratify=bc_data[target]
)

predictor = TabularPredictor(
    label=target,
    eval_metric="f1_weighted"
).fit(
    train_data=train_df,
    presets="high_quality",
    time_limit=1800
)

pred = predictor.predict(test_df)

print("Accuracy:", accuracy_score(test_df[target], pred))
print("F1:", f1_score(test_df[target], pred))

No path specified. Models will be saved in: "AutogluonModels/ag-20260728_210634"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.87 GB / 12.67 GB (85.8%)
Disk Space Avail:   58.66 GB / 112.64 GB (52.1%)
Presets specified: ['high_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will 

Accuracy: 0.9940571428571429


ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].

In [10]:
!pip install flaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.7/349.7 kB 8.1 MB/s eta 0:00:00


In [13]:
from sklearn.model_selection import train_test_split
from flaml import AutoML
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


X = bc_data.drop(columns=[target])
y = bc_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

automl = AutoML()

settings = {
    "time_budget": 1800,
    "metric": "accuracy",      # or "macro_f1" if supported by your FLAML version
    "task": "classification",
    "seed": 42,
}

automl.fit(
    X_train=X_train,
    y_train=y_train,
    **settings
)

pred = automl.predict(X_test)

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred, average="weighted"))
print("Recall   :", recall_score(y_test, pred, average="weighted"))
print("F1 Score :", f1_score(y_test, pred, average="weighted"))

print("\nBest Model:", automl.best_estimator)
print("Best Config:", automl.best_config)

[flaml.automl.logger: 07-28 22:03:14] {2375} INFO - task = classification
[flaml.automl.logger: 07-28 22:03:14] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 07-28 22:03:14] {2489} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 07-28 22:03:14] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'catboost', 'lrl1']
[flaml.automl.logger: 07-28 22:03:14] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 07-28 22:03:16] {3046} INFO - Estimated sufficient time budget=18723s. Estimated necessary time budget=462s.
[flaml.automl.logger: 07-28 22:03:16] {3097} INFO -  at 2.0s,	estimator lgbm's best error=9.4171e-02,	best estimator lgbm's best error=9.4171e-02
[flaml.automl.logger: 07-28 22:03:16] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 07-28 22:03:17] {3097} INFO -  at 3.3s,	estimator lgbm's best error=9.4171e-02,	best estimator lgbm's best error=9.4171e-

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 07-28 22:04:04] {3097} INFO -  at 50.2s,	estimator sgd's best error=1.9909e-01,	best estimator lgbm's best error=3.5048e-02
[flaml.automl.logger: 07-28 22:04:04] {2911} INFO - iteration 4, current learner lgbm
[flaml.automl.logger: 07-28 22:04:08] {3097} INFO -  at 54.7s,	estimator lgbm's best error=2.5486e-02,	best estimator lgbm's best error=2.5486e-02
[flaml.automl.logger: 07-28 22:04:08] {2911} INFO - iteration 5, current learner xgboost
[flaml.automl.logger: 07-28 22:04:10] {3097} INFO -  at 56.3s,	estimator xgboost's best error=9.5962e-02,	best estimator lgbm's best error=2.5486e-02
[flaml.automl.logger: 07-28 22:04:10] {2911} INFO - iteration 6, current learner lgbm
[flaml.automl.logger: 07-28 22:04:11] {3097} INFO -  at 57.5s,	estimator lgbm's best error=2.5486e-02,	best estimator lgbm's best error=2.5486e-02
[flaml.automl.logger: 07-28 22:04:11] {2911} INFO - iteration 7, current learner lgbm
[flaml.automl.logger: 07-28 22:04:17] {3097} INFO -  at 62.9s,	